In [31]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error as MSE
from sklearn.linear_model import LinearRegression
import warnings
import os 

warnings.filterwarnings('ignore')

In [32]:
df = pd.read_csv('../data/processed/train_features.csv')

print('Shape', df.shape)
print('Columns', df.columns.tolist())
df.head()

Shape (844338, 20)
Columns ['Store', 'DayOfWeek', 'Sales', 'Customers', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'Year', 'Month', 'Day', 'Week', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd', 'CompetitionOpen', 'Promo2Active']


,Store,DayOfWeek,Sales,Customers,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,Promo2,Year,Month,Day,Week,IsWeekend,IsMonthStart,IsMonthEnd,CompetitionOpen,Promo2Active
0,1,5,5263,555,1,0,1,2,0,1270.0,0,2015,7,31,31,0,0,1,82.0,0
1,2,5,6064,625,1,0,1,0,0,570.0,1,2015,7,31,31,0,0,1,92.0,1
2,3,5,8314,821,1,0,1,0,0,14130.0,1,2015,7,31,31,0,0,1,103.0,1
3,4,5,13995,1498,1,0,1,2,2,620.0,0,2015,7,31,31,0,0,1,70.0,0
4,5,5,4822,559,1,0,1,0,0,29910.0,0,2015,7,31,31,0,0,1,3.0,0


In [33]:
TARGET = 'Sales'

FEATURES = df.columns.tolist()

FEATURES.remove("Sales")
FEATURES.remove('Customers')

print(f"Target: {TARGET}")
print(f'Features: {FEATURES}')
print(f"Features Length: {len(FEATURES)}")

Target: Sales
Features: ['Store', 'DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'Year', 'Month', 'Day', 'Week', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd', 'CompetitionOpen', 'Promo2Active']
Features Length: 18


In [34]:
df = df.sort_values(['Year', 'Week']).reset_index(drop=True)

split_idx = int(len(df) * 0.85)

train_df = df.iloc[:split_idx]
val_df = df.iloc[split_idx:]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val = val_df[FEATURES]
y_val = val_df[TARGET]

print(f'Train size: {len(train_df):,}')
print(f'Test size: {len(val_df):,}')
print(f'Trin % {len(train_df) / len(df) * 100 : .1f}%')

Train size: 717,687
Test size: 126,651
Trin %  85.0%


In [35]:
def rmspe(y_true, y_pred):
    mask = y_true != 0
    return np.sqrt(np.mean(( ( y_true[mask] - y_pred[mask] ) / y_true[mask] ) ** 2))

In [36]:
mean_sales = y_train.mean()
baseline_preds = np.full(len(y_val), mean_sales)

baseline_rmspe = rmspe(y_val.values, baseline_preds)

print(f'Mean sales (train): {mean_sales:.2f}')
print(f'Baseline RMSPE: {baseline_rmspe:.4f}')

Mean sales (train): 6904.23
Baseline RMSPE: 0.5445


In [37]:
lr = LinearRegression()
lr.fit(X_train, y_train)

lr_preds = lr.predict(X_val)
lr_preds = np.maximum(lr_preds, 0)

lr_rmspe = rmspe(y_val.values, lr_preds)

print(f"Linear Regression RMSPE: {lr_rmspe:.4f}")

Linear Regression RMSPE: 0.4613


In [40]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbose': -1
}

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=100)
]

lgb_model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_val],
    callbacks=callbacks
)

lgb_preds = lgb_model.predict(X_val)
lgb_preds = np.maximum(lgb_preds, 0)

lgb_rmspe = rmspe(y_val.values, lgb_preds)

print(f"\nLightGBM RMSPE: {lgb_rmspe:.4f}")
print(f"Best iteration: {lgb_model.best_iteration}")

[100]	valid_0's rmse: 2010.92
[200]	valid_0's rmse: 1689.63
[300]	valid_0's rmse: 1512.65
[400]	valid_0's rmse: 1420.58
[500]	valid_0's rmse: 1358.66
[600]	valid_0's rmse: 1312.29
[700]	valid_0's rmse: 1285.51
[800]	valid_0's rmse: 1258.99
[900]	valid_0's rmse: 1241.9
[1000]	valid_0's rmse: 1229.21

LightGBM RMSPE: 0.1852
Best iteration: 998


In [42]:
results = pd.DataFrame({
    'Models': ['Mean Baseline', 'Linear regression', 'LightGBM'],
    'RMSPE': [baseline_rmspe, lr_rmspe, lgb_rmspe]
}).sort_values('RMSPE')

results["RMSPE"] = results["RMSPE"].round(4)
results["vs Baseline"] = ((results["RMSPE"] - baseline_rmspe) / baseline_rmspe * 100).round(1).astype(str) + "%"

print(results.to_string(index=False))


           Models  RMSPE vs Baseline
         LightGBM 0.1852      -66.0%
Linear regression 0.4613      -15.3%
    Mean Baseline 0.5445       -0.0%


In [44]:
importance = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': lgb_model.feature_importance(importance_type='gain')
}).sort_values('Importance', ascending=False)


print(importance.to_string(index=False))

            Feature   Importance
              Store 1.306401e+13
CompetitionDistance 1.287477e+13
              Promo 7.403334e+12
    CompetitionOpen 3.516519e+12
          DayOfWeek 3.221337e+12
          StoreType 2.825415e+12
         Assortment 2.195584e+12
                Day 1.903735e+12
               Week 1.891003e+12
             Promo2 1.536618e+12
              Month 9.567295e+11
               Year 2.148479e+11
          IsWeekend 1.754987e+11
         IsMonthEnd 1.650718e+11
      SchoolHoliday 1.271861e+11
       StateHoliday 5.168940e+10
       Promo2Active 5.046845e+10
       IsMonthStart 1.322812e+09


In [45]:
os.makedirs("../models", exist_ok=True)

lgb_model.save_model("../models/lgb_model.txt")

print("Model saved to ../models/lgb_model.txt")

Model saved to ../models/lgb_model.txt


## Modelling Summary

### Train/validation split
- Sorted by Year then Week to ensure chronological order
- 85% train (717,687 rows) / 15% validation (126,651 rows)
- Validation covers the most recent period — mimics real deployment

### Results

| Model | RMSPE | vs Baseline |
|-------|-------|-------------|
| LightGBM | 0.1852 | -66.0% |
| Linear Regression | 0.4613 | -15.3% |
| Mean Baseline | 0.5445 | — |

### Top 5 features by importance (gain)
1. Store — individual store identity dominates
2. CompetitionDistance — proximity to competitor matters
3. Promo — daily promotion is a strong sales driver
4. CompetitionOpen — months since competitor opened
5. DayOfWeek — day patterns are significant

### Observations
- LightGBM ran to iteration 998 (early stopping at 50 rounds not triggered)
- Model had not fully converged — increasing num_boost_round to 2000 may improve score
- Promo2Active ranked low — Promo2 effect may be weaker than daily Promo
- Store as top feature suggests per-store baselines would be a strong addition

### Next step
Evaluation notebook — error analysis, SHAP values, business interpretation